In [0]:
%run /Users/sandysakthivel2005@gmail.com/common/03_Logger

2026-07-10 04:38:56,117 | INFO | Received command c on object id p1


Logger notebook executed successfully


In [0]:
print(logger)

2026-07-10 04:39:03,625 | INFO | Received command c on object id p1


<Logger SocialMediaPipeline (INFO)>


In [0]:
# Databricks notebook source

# MAGIC %run /Users/sandysakthivel2005@gmail.com/common/03_Logger

from pyspark.sql.functions import *

try:

    logger.info("Silver User Metadata Pipeline Started")
    print("Silver User Metadata Pipeline Started")

    # ==========================================
    # Read Bronze Streaming Table
    # ==========================================

    bronzeDF = (
        spark.readStream
             .table("bronze_catalog1.raw.bronze_user_metadata1")
    )

    logger.info("Bronze User Metadata Table Read Successfully")
    print("Bronze User Metadata Table Read Successfully")

    # ==========================================
    # Remove Duplicates
    # ==========================================

    silverDF = bronzeDF.dropDuplicates(["user_id"])

    # ==========================================
    # Handle Null Values
    # ==========================================

    silverDF = (
        silverDF
            .fillna({
                "country": "Unknown",
                "topic_category": "Unknown",
                "followers_count": 0,
                "following_count": 0,
                "likes_count": 0,
                "shares_count": 0,
                "posts_count": 0,
                "verified": "No"
            })
    )

    # ==========================================
    # Data Validation
    # ==========================================

    silverDF = (
        silverDF
            .filter(col("user_id").isNotNull())
            .filter(col("country").isNotNull())
            .filter(col("topic_category").isNotNull())
            .filter(col("account_created_date").isNotNull())
            .filter(col("followers_count") >= 0)
            .filter(col("following_count") >= 0)
            .filter(col("likes_count") >= 0)
            .filter(col("shares_count") >= 0)
            .filter(col("posts_count") >= 0)
    )

    # ==========================================
    # Standardize Text
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("country", upper(trim(col("country"))))
            .withColumn("topic_category", upper(trim(col("topic_category"))))
            .withColumn("verified", upper(trim(col("verified"))))
    )

    # ==========================================
    # Convert Data Types
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("account_created_date", to_date(col("account_created_date")))
            .withColumn("followers_count", col("followers_count").cast("int"))
            .withColumn("following_count", col("following_count").cast("int"))
            .withColumn("likes_count", col("likes_count").cast("int"))
            .withColumn("shares_count", col("shares_count").cast("int"))
            .withColumn("posts_count", col("posts_count").cast("int"))
    )

    # ==========================================
    # Audit Columns
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("silver_load_time", current_timestamp())
            .withColumn("pipeline_name", lit("Silver_User_Metadata"))
    )

    logger.info("Silver User Metadata Transformations Completed Successfully")
    print("Silver User Metadata Transformations Completed Successfully")

    # ==========================================
    # Write Silver Table
    # ==========================================

    silverQuery = (
        silverDF.writeStream
            .trigger(availableNow=True)
            .format("delta")
            .outputMode("append")
            .option(
                "checkpointLocation",
                "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/silver_user_metadata"
            )
            .option("mergeSchema", "true")
            .toTable("silver_catalog1.processed.silver_user_metadata")
    )

    silverQuery.awaitTermination()

    logger.info("Silver User Metadata Loaded Successfully")
    print("Silver User Metadata Loaded Successfully")

except Exception as e:

    logger.error(f"Silver User Metadata Pipeline Failed: {str(e)}")
    print(f"Silver User Metadata Pipeline Failed: {str(e)}")

    raise

2026-07-10 04:39:07,091 | INFO | Received command c on object id p1
2026-07-10 04:39:07,113 | INFO | Silver User Metadata Pipeline Started
2026-07-10 04:39:07,114 | INFO | Error while sending or receiving.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
2026-07-10 04:39:07,115 | INFO | Closing down clientserver connection
2026-07-10 04:39:07,116 | INFO | Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-

Silver User Metadata Pipeline Started


2026-07-10 04:39:08,231 | INFO | Received command c on object id p0
2026-07-10 04:39:09,231 | INFO | Received command c on object id p0
2026-07-10 04:39:10,231 | INFO | Received command c on object id p0
2026-07-10 04:39:11,231 | INFO | Received command c on object id p0
2026-07-10 04:39:12,231 | INFO | Received command c on object id p0
2026-07-10 04:39:13,068 | INFO | Bronze User Metadata Table Read Successfully
2026-07-10 04:39:13,231 | INFO | Received command c on object id p0
2026-07-10 04:39:13,254 | INFO | Error while sending or receiving.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
2026-07-10 04:39:13,256 | INFO | Closing down clientserver connection
2026-07-10 04:39:13,257 | INFO | Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark

Bronze User Metadata Table Read Successfully


2026-07-10 04:39:14,001 | INFO | Silver User Metadata Transformations Completed Successfully


Silver User Metadata Transformations Completed Successfully


2026-07-10 04:39:14,231 | INFO | Received command c on object id p0
2026-07-10 04:39:15,231 | INFO | Received command c on object id p0
2026-07-10 04:39:16,231 | INFO | Received command c on object id p0
2026-07-10 04:39:17,231 | INFO | Received command c on object id p0
2026-07-10 04:39:18,231 | INFO | Received command c on object id p0
2026-07-10 04:39:19,231 | INFO | Received command c on object id p0
2026-07-10 04:39:20,231 | INFO | Received command c on object id p0
2026-07-10 04:39:21,231 | INFO | Received command c on object id p0
2026-07-10 04:39:22,231 | INFO | Received command c on object id p0
2026-07-10 04:39:23,231 | INFO | Received command c on object id p0
2026-07-10 04:39:24,294 | INFO | Received command c on object id p0
2026-07-10 04:39:25,012 | INFO | Silver User Metadata Loaded Successfully


Silver User Metadata Loaded Successfully


In [0]:
%sql
SHOW TABLES IN silver_catalog1.processed;

database,tableName,isTemporary
processed,silver_sentiment,false
processed,silver_trends,false
processed,silver_tweets,false
processed,silver_tweets_clean,false
processed,silver_user_metadata,false


In [0]:
%sql
SELECT COUNT(*)
FROM silver_catalog1.processed.silver_user_metadata;

count(1)
2329


In [0]:
%sql
SELECT *
FROM silver_catalog1.processed.silver_user_metadata
LIMIT 20;

user_id,country,topic_category,account_created_date,followers_count,following_count,likes_count,shares_count,posts_count,verified,bronze_load_time,pipeline_name,source_system,ingestion_date,silver_load_time
6918.0,GERMANY,TECH,2025-01-21,75039,4590,1629,24,656,"""NAN""",2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
8799.0,INDIA,TECH,2025-01-01,76495,2815,1252,266,418,N,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
8850.0,USA,SPORTS,2025-01-11,7320,3943,203,101,158,Y,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
8348.0,CANADA,POLITICS,2025-01-19,80766,1463,1578,21,334,Y,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
9160.0,INDIA,POLITICS,2025-01-20,68480,3026,430,260,675,N,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
7838.0,INDIA,FINANCE,2025-01-10,62740,3698,748,0,397,N,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
7797.0,USA,SPORTS,2025-01-08,36705,4062,1521,488,857,N,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
9983.0,UK,SPORTS,2025-01-13,12408,176,1272,265,711,Y,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
8337.0,UK,SPORTS,2025-01-06,0,4081,966,47,510,N,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
4928.0,INDIA,"""NAN""",2025-01-02,34269,835,293,372,832,Y,2026-07-09T09:35:02.966Z,Silver_User_Metadata,Azure Event Hub,2026-07-09,2026-07-10T04:31:35.112Z
